# Parallelizing with JAX
In a previous notebook, we've explored how to get started with JAX on TIKE. We've seen already that we can dramatically accelerate our code with even relatively simple changes to our code.

One of the benefits of using TIKE is that we have access to multiple cores. How can we use JAX to most efficiently use all these cores? Let's find out and make sure we're *parallelizing* correctly!

First, let's again make sure JAX is installed in this kernel.

In [2]:
!pip install -U jax

  Using cached ml_dtypes-0.5.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.2/78.2 MB 105.4 MB/s eta 0:00:0000:0100:01
Using cached ml_dtypes-0.5.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (4.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 47.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4━━━━━ 0/4 [numpy]
  Attempting uninstall: ml_dtypes━━━━━━━━━━━━━━━ 0/4 [numpy]
    Found existing installation: ml-dtypes 0.3.2 0/4 [numpy]
    Uninstalling ml-dtypes-0.3.2:━━━━━━━━━━━ 0/4 [numpy]
      Successfully uninstalled ml-dtypes-0.3.2━━━━━━━━━━━━━━━━━━━━ 1/4 [ml_dtypes]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [jax]3/4 [jax]ib]
ERROR: pip's dependency resolver does not cur

In [5]:
!pip install --upgrade lightkurve

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.3 which is incompatible.


In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import lightkurve as lk

# Implicit parallelization with JAX

To explore how JAX parallelizes things, let's first instantiate a large array.

As you run these computations, watch the "kernel usage" in the tab to the right of your screen, in particular the "Host CPU".

In [2]:
big_array = jnp.ones((10000, 10000))

Let's dot this array with itself, timing how long the operation takes.

In [3]:
%%time
jnp.dot(big_array, big_array).block_until_ready()

CPU times: user 27 s, sys: 292 ms, total: 27.3 s
Wall time: 7.27 s


Array([[10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       ...,
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.]],      dtype=float32)

Let's do the same with a `numpy` array.

In [5]:
big_array = np.ones((10000, 10000))

In [53]:
%%time
np.dot(big_array, big_array)

CPU times: user 49.8 s, sys: 2.21 s, total: 52 s
Wall time: 13.9 s


array([[10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       ...,
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.]])

In both cases, we were able to use 100% of our CPUs. This seems a bit like magic — we didn't even explictly request any parallelization! Note, of course, that the jax implementation was faster. This is because JAX is friendlier to the device used.

It turns out, both `jax` and vanilla `numpy` do perform some parallelization on array operations under the hood. `Numpy` operations such as `np.dot` [utilize calls to the BLAS routines](https://numpy.org/doc/stable/reference/generated/numpy.dot.html), which are essentially a large number of linear algebra routines implemented in Fortran and C. BLAS automatically recognizes the number of cores available in your current system and will create multiple *threads* to take advantage of the these cores.

`JAX` operates similarly, with *its* linear algebra engine (XLA) making use of the available local cores, as well.

# Vectorization with JAX
There's another clear way in which we can take advantage of your hardware to speed up code. That's through something called *vectorization*.

Essentially, vectorization eliminates the inefficiencies of explicit loops. Modern computer hardware often has the ability to perform the same operation multiple times on an array, so long as it is prompted to do so. Normally, loops don't take advantage of this, while vectorized code can. 

Writing vectorized code can be tricky, though. It requires keeping strict track of, for instance, array dimensions.

Luckily, JAX has a feature known as "autovectorization." Because JAX has the ability to trace through your Python functions and fuse operations together, it can automatically recognize which portions of your code can be vectorized.

Let's walk through how to do this. First, let's write a function that operates on two arrays. It takes all the values of the first array and multiplies them by the maximum of the second array.

In [20]:
def array_op(array1, array2):
    max_val = jnp.max(array2)
    return array1 * max_val

Let's make a few arrays now!

In [21]:
array1 = jnp.arange(25)
array2 = jnp.linspace(-29, 20, 4)

Now, let's time our operation. Whenever we perform these timing experiments, we should be careful to use the `block_until_ready` method on anything containing JAX array objects. The reason for this is that JAX has asynchronous dispatch — i.e., Python code can continue past a JAX computation before the JAX computation finishes. This behavior is useful for accelerating programs, but it's not as useful for benchmarking. `block_until_ready` ensures that a program doesn't continue until the computation has actually completed.

In [29]:
%%timeit
array_op(array1, array2).block_until_ready()

13.6 μs ± 263 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Now, let's stack some arrays and manually loop over them to perform our calculation. Note that this is the inefficient way to do it!

In [30]:
array1s = jnp.stack([array1, array1, array1])
array2s = jnp.stack([array2, array2, array2])

In [31]:
%%timeit
for array1, array2 in zip(array1s, array2s):
    array_op(array1, array2).block_until_ready()

60.3 μs ± 1.4 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Now, let's vectorize instead.

In [34]:
auto_array_op = jax.vmap(array_op)

In [35]:
%%timeit
auto_array_op(array1s, array2s).block_until_ready()

442 μs ± 10.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


What gives? This is slower than before!

The reason is that `JAX`'s `vmap` performs well when the batch dimension is large. Let's try with a larger dimension.

In [47]:
batch_size = 1000
array1s = jnp.stack([array1] * batch_size)   # shape: (1000, 25)
array2s = jnp.stack([array2] * batch_size)   # shape: (1000, 4)

In [48]:
%%timeit
for array1, array2 in zip(array1s, array2s):
    array_op(array1, array2).block_until_ready()

24.2 ms ± 1.67 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [49]:
%%timeit
auto_array_op(array1s, array2s).block_until_ready()

494 μs ± 15.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Wow! The vectorized version scales very well.

We can also compose `JAX` functions together. Recall that `jit` can speed things up quite a bit; let's try that.

In [50]:
jitted_array_op = jax.jit(auto_array_op)

jitted_array_op(array1s, array2s)

Array([[  0.,  20.,  40., ..., 440., 460., 480.],
       [  0.,  20.,  40., ..., 440., 460., 480.],
       [  0.,  20.,  40., ..., 440., 460., 480.],
       ...,
       [  0.,  20.,  40., ..., 440., 460., 480.],
       [  0.,  20.,  40., ..., 440., 460., 480.],
       [  0.,  20.,  40., ..., 440., 460., 480.]], dtype=float32)

In [52]:
%%timeit
jitted_array_op(array1s, array2s).block_until_ready()

30.1 μs ± 814 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


This is almost 3 orders of magnitude faster than the non-vectorized version — with minimal changes to our code!

# When parallelizing with JAX does not work as well

We've just shown a few ways in which parallelization with JAX requires (relatively) minor changes to our code. This is in part because we've hand-picked a few suitable examples! There are certainly cases in which parallelization with JAX is not as useful.

For instance: if your computation is very string-heavy (e.g., concatenating large strings), JAX is less useful, because it cannot vectorize string operations. Similarly, for sequential I/O type operations or for functions with "side effects" (e.g., writing logs), these types of operations cannot be performed in parallel. Finally, if your function cannot be traced by JAX (e.g., it includes irregular Python classes), JAX can't speed it up.

In more straightforward cases, though — numeric calculations on dense, numeric, fixed-shape arrays, especially larger ones — JAX should be able to help.

# Basic science case: differencing a lightcurve
Let's perform some basic parallelization by vectorizing the first few steps of a flare detection pipeline: taking the difference of subsequent values in a lighcurve. Let's start by downloading a Kepler lightcurve. To make this example amenable to vectorization, let's download multiple quarters of data.

In [11]:
search_result = lk.search_lightcurve('KIC 3733346', author='Kepler')
search_result

#,mission,year,author,exptime,target_name,distance
,,,,s,,arcsec
0,Kepler Quarter 01,2009,Kepler,1800,kplr003733346,0.0
1,Kepler Quarter 02,2009,Kepler,1800,kplr003733346,0.0
2,Kepler Quarter 03,2009,Kepler,1800,kplr003733346,0.0
3,Kepler Quarter 04,2010,Kepler,1800,kplr003733346,0.0
4,Kepler Quarter 05,2010,Kepler,1800,kplr003733346,0.0
5,Kepler Quarter 06,2010,Kepler,1800,kplr003733346,0.0
6,Kepler Quarter 07,2010,Kepler,1800,kplr003733346,0.0
7,Kepler Quarter 11,2011,Kepler,60,kplr003733346,0.0
8,Kepler Quarter 10,2011,Kepler,1800,kplr003733346,0.0


In [13]:
lcs = search_result.download_all()

In [36]:
flux_arrays = [jnp.array(lc.flux.value) for lc in lcs]

In [37]:
flux_arrays

[Array([94774.3  , 87980.36 , 90003.555, ..., 95417.43 , 97040.45 ,
        96912.66 ], dtype=float32),
 Array([92319.516, 89214.01 , 85608.195, ..., 92305.73 , 89541.5  ,
        85948.555], dtype=float32),
 Array([      nan, 112645.  , 109691.59, ..., 136450.6 , 132225.11,
        128716.43], dtype=float32),
 Array([      nan, 107816.1 , 105917.04, ..., 149477.34, 143782.16,
        138811.52], dtype=float32),
 Array([       nan, 144445.77 , 138555.5  , ...,  89414.414,  87076.24 ,
        101960.48 ], dtype=float32),
 Array([84514.87 , 82686.14 , 85710.805, ..., 98026.72 , 96062.14 ,
        92780.82 ], dtype=float32),
 Array([      nan, 91261.39 , 91528.945, ..., 99770.12 , 96013.664,
        90341.9  ], dtype=float32),
 Array([      nan,  92241.93,  92124.23, ..., 113820.12, 113795.39,
        113601.72], dtype=float32),
 Array([       nan,  92746.055,  92672.695, ..., 102448.02 ,  99558.44 ,
         96988.97 ], dtype=float32),
 Array([      nan, 113897.1 , 110616.16, ..., 156583

A tricky data consideation is that all of these lightcurves have different sizes, so we can't stack them and nicely vectorize them. Let's pad the shorter arrays with 0s.

In [44]:
[len(arr) for arr in flux_arrays]

[1626,
 4075,
 4140,
 4116,
 4492,
 4276,
 4229,
 45194,
 4447,
 4618,
 3113,
 4480,
 3551,
 4252,
 4344,
 4446,
 3540,
 1286]

In [46]:
lengths = [len(arr) for arr in flux_arrays]
max_length = np.max(lengths)
max_length

45194

Now, let's pad on just one side.

In [49]:
flux_arrays_padded = jnp.array([jnp.pad(flux_array, (0, max_length - length))
                                for flux_array, length in zip(flux_arrays, lengths)])

Now, let's perform a simple vectorized operation.

In [54]:
def array_diff(arr):
    diff_val = jnp.diff(arr)
    return diff_val

In [55]:
auto_array_diff = jax.vmap(array_diff)

In [58]:
%%timeit
res = auto_array_diff(flux_arrays_padded)
res.block_until_ready()

500 μs ± 26.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Nice! This was quite fast. Let's see how fast it is compared to a non-vectorized version.

In [59]:
%%timeit
for i in range(len(flux_arrays_padded)):
    array_diff(flux_arrays_padded[i])

2.61 ms ± 187 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


The vectorized version is roughly 4x faster. Nice!